# Checkpoint 3 v3

## the trainer class

In [ ]:
class ANITrainer:
    def __init__(self, model, batch_size, learning_rate, epoch, l2):
        self.model = model
        
        num_params = sum(item.numel() for item in model.parameters())
        print(f"{model.__class__.__name__} - Number of parameters: {num_params}")
        
        self.batch_size = batch_size #...
        self.optimizer = optim.Adam(self.model.parameters(),lr=learning_rate,weight_decay=l2) #...
        self.epoch = epoch #...
    
    def train(self, train_data, val_data, early_stop=True, draw_curve=True): #early stopping! :)
        self.model.train()
        
        # init data loader
        print("Initialize training data...")
        #train_data_loader = DataLoader(train_data,batch_size=self.batch_size,shuffle=True) #...
        train_data_loader = train_data.collate(self.batch_size).cache()

        # definition of loss function: MSE is a good choice! 
        loss_func = torch.nn.MSELoss() #...
        
        # record epoch losses
        train_loss_list = []
        val_loss_list = []
        lowest_val_loss = np.inf
        
        for i in tqdm(range(self.epoch), leave=True):
            train_epoch_loss = 0.0
            for train_data_batch in train_data_loader:
                
                # compute energies #...
                # (self) assuming train data batch has inputs and true_energies
                #inputs, true_energies=train_data_batch
                species = train_data_batch['species'].to(device)
                coordinates = train_data_batch['coordinates'].to(device)
                true_energies = train_data_batch['energies'].to(device).float()

                _, pred_energies = self.model((species, coordinates))
                
                #pred_energies=self.model(inputs)
                
                # compute loss
                batch_loss = loss_func(pred_energies,true_energies) #...
                
                # do a step #...
                self.optimizer.zero_grad()
                batch_loss.backward()
                self.optimizer.step()
                
                batch_importance = 1 #... this assumes all batches are equally impt - double check
                train_epoch_loss += batch_loss.item()*batch_importance #...
            
            # use the self.evaluate to get loss on the validation set 
            val_epoch_loss, _ = self.evaluate(val_data,draw_plot=False) #...
            
            # append the losses
            #...
            train_loss_list.append(train_epoch_loss/len(train_data_loader))
            val_loss_list.append(val_epoch_loss)
            
            if early_stop:
                if val_epoch_loss < lowest_val_loss:
                    lowest_val_loss = val_epoch_loss
                    weights = self.model.state_dict()
        
        if draw_curve:
            fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
            ax.set_yscale("log")
            # Plot train loss and validation loss
            ax.plot(range(self.epoch),train_loss_list, label='Train')
            ax.plot(range(self.epoch),val_loss_list, label='Validation')
            ax.legend()
            ax.set_xlabel("# Batch")
            ax.set_ylabel("Loss")
        
        if early_stop:
            self.model.load_state_dict(weights)
        
        return train_loss_list, val_loss_list
    
    
    def evaluate(self, data, draw_plot=False):
        
        # init data loader
        # data_loader = DataLoader(data,batch_size=self.batch_size,shuffle=False) #...
        data_loader = data.collate(self.batch_size).cache()
        
        # init loss function
        loss_func = torch.nn.MSELoss() #...
        total_loss = 0.0
        
        #if draw_plot:
        true_energies_all = []
        pred_energies_all = []
            
        with torch.no_grad():
            for batch_data in data_loader:
                
                # compute energies #...
                #inputs,true_energies=batch_data
                #pred_energies=self.model(inputs)
                species = batch_data['species'].to(device)
                coordinates = batch_data['coordinates'].to(device)
                true_energies = batch_data['energies'].to(device).float()

                _, pred_energies = self.model((species, coordinates))
                
                # compute loss
                batch_loss = loss_func(pred_energies,true_energies) #...

                batch_importance = 1 #...
                total_loss += batch_loss.item()*batch_importance #...
                
                #if draw_plot:
                true_energies_all.append(true_energies.detach().cpu().numpy().flatten())
                pred_energies_all.append(pred_energies.detach().cpu().numpy().flatten())
                    
        true_energies_all = np.concatenate(true_energies_all)
        pred_energies_all = np.concatenate(pred_energies_all)
        hartree2kcalmol = 627.5094738898777
        mae = np.mean(np.abs(true_energies_all - pred_energies_all))*hartree2kcalmol

        if draw_plot:
            #true_energies_all = np.concatenate(true_energies_all)
            #pred_energies_all = np.concatenate(pred_energies_all)
            # Report the mean absolute error
            # The unit of energies in the dataset is hartree
            # please convert it to kcal/mol when reporting the mean absolute error
            # 1 hartree = 627.5094738898777 kcal/mol
            # MAE = mean(|true - pred|)
            #hartree2kcalmol = 627.5094738898777 #...
            #mae = np.mean(np.abs(true_energies_all - pred_energies_all))*hartree2kcalmol #... in kcal/mol 
            fig, ax = plt.subplots(1,1,figsize=(5,4),constrained_layout=True)
            ax.scatter(true_energies_all, pred_energies_all, label=f"MAE: {mae:.2f} kcal/mol", s=2)
            ax.set_xlabel("Ground Truth")
            ax.set_ylabel("Predicted")
            xmin, xmax = ax.get_xlim()
            ymin, ymax = ax.get_ylim()
            vmin, vmax = min(xmin, ymin), max(xmax, ymax)
            ax.set_xlim(vmin, vmax)
            ax.set_ylim(vmin, vmax)
            ax.plot([vmin, vmax], [vmin, vmax], color='red')
            ax.legend()
            
        return total_loss, mae

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
aev_computer = init_aev_computer() 

aev_dim = aev_computer.aev_length
species_order = ['H', 'C', 'N', 'O']
dataset = torchani.data.load('./ani_gdb_s01_to_s04.h5').subtract_self_energies(
    utils.EnergyShifter(None), species_order
).species_to_indices(species_order).shuffle()
train_data, val_data, test_data = dataset.split(0.8, 0.1, 0.1)

# experimenting with model
def atomic_model(aev_computer,hidden_sizes=[128,128,64],dropout=0.0,device='cuda'):
    def build_net():
        layers = []
        input_dim = aev_computer.aev_length
        for h in hidden_sizes:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        return nn.Sequential(*layers)
    
    return nn.Sequential(
        aev_computer,
        torchani.ANIModel([
            build_net(),  # H
            build_net(),  # C
            build_net(),  # N
            build_net()   # O
        ])
    ).to(device)
    
# hyperparameter tuning stuff
from itertools import product

hiddens = [[128,128,64],[256,128],[128,128,128,64]]
dropouts = [0.0,0.1]
lr_options = [1e-3,5e-4]
l2_options = [0.0,1e-4]

results = []

for h, d, lr, l2 in itertools.product(hiddens,dropouts,lr_options,l2_options):
    print(f"\nTraining: hidden={h}, dropout={d}, lr={lr}, l2={l2}")
    
    model = atomic_model(aev_computer, hidden_sizes=h, dropout=d, device=device)

    trainer = ANITrainer(model=model,batch_size=1024,learning_rate=lr,epoch=5,l2=l2)
    
    train_loss_list, val_loss_list = trainer.train(train_data, val_data, draw_curve=False)
    val_loss, val_mae = trainer.evaluate(val_data, draw_plot=False)

    results.append({
        "hidden": h,
        "dropout": d,
        "lr": lr,
        "l2": l2,
        "val_loss": val_loss,
        "mae": val_mae,
        "train_curve": train_loss_list,
        "val_curve": val_loss_list
    })

# final model choice
best = sorted(results, key=lambda r: r['mae'])[0]
print("\nBest model final:")
print(best)


Training: hidden=[128, 128, 64], dropout=0.0, lr=0.001, l2=0.0
Sequential - Number of parameters: 296452
Initialize training data...


100%|██████████| 5/5 [01:00<00:00, 12.14s/it]



Training: hidden=[128, 128, 64], dropout=0.0, lr=0.001, l2=0.0001
Sequential - Number of parameters: 296452
Initialize training data...


100%|██████████| 5/5 [00:51<00:00, 10.37s/it]



Training: hidden=[128, 128, 64], dropout=0.0, lr=0.0005, l2=0.0
Sequential - Number of parameters: 296452
Initialize training data...


...


Training: hidden=[128, 128, 128, 64], dropout=0.1, lr=0.0005, l2=0.0
Sequential - Number of parameters: 362500
Initialize training data...


100%|██████████| 5/5 [01:00<00:00, 12.12s/it]



Training: hidden=[128, 128, 128, 64], dropout=0.1, lr=0.0005, l2=0.0001
Sequential - Number of parameters: 362500
Initialize training data...


100%|██████████| 5/5 [01:01<00:00, 12.28s/it]



Best model final:
{'hidden': [256, 128], 'dropout': 0.0, 'lr': 0.0005, 'l2': 0.0, 'val_loss': 0.0005820179089823796, 'mae': 1.18688914949423, 'train_curve': [0.00031076587544794706, 1.5794090143857475e-05, 1.6407515735027012e-05, 1.3378315622756515e-05, 1.1862248993296946e-05], 'val_curve': [0.0018589490064186975, 0.0014285133565863362, 0.0009099460503421142, 0.0007058740943648445, 0.0005820179085276322]}


Chose epoch=5 to ensure validation curve was decreasing as batch number increases. When epoch =1 or 2, the validation curve plotted was too eratic - indicating that perhaps training could be improved.

The best-performing model used the hidden layers [256,128] with no dropout and L2 regularization, and a learning rate of 0.0005. MAE=1.1 kcal/mol. For my final model choice, I plan to use 10-15 epochs to better train it.

In [18]:
print(best.keys())
print("Train curve:",best['train_curve'])
print("Val curve:",best['val_curve'])

dict_keys(['hidden', 'dropout', 'lr', 'l2', 'val_loss', 'mae', 'train_curve', 'val_curve'])
Train curve: [0.00031076587544794706, 1.5794090143857475e-05, 1.6407515735027012e-05, 1.3378315622756515e-05, 1.1862248993296946e-05]
Val curve: [0.0018589490064186975, 0.0014285133565863362, 0.0009099460503421142, 0.0007058740943648445, 0.0005820179085276322]
